# Docebo Training Material Automation
Auto-fill data from Excel into Docebo LMS (Deferred sheet)

In [ ]:
import sys
!{sys.executable} -m pip install selenium pandas openpyxl -q
print("Done")

In [7]:
EXCEL_FILE = r"C:\Users\JiunYong_Liang\OneDrive - Dell Technologies\Desktop\Dell_TM_Migration_Report_updated06032026.xlsx"
SHEET_NAME = "Deferred"
DOCEBO_BASE = "https://dellsandbox.docebosaas.com"
SESSION_DIR = r"C:\Users\JiunYong_Liang\AppData\Local\playwright_docebo_session"
START_ROW = 604   # Excel row 527, ESCPXD08326
END_ROW = 644     # Excel row 550 (inclusive)

In [8]:
import shutil, pandas as pd

dst = r"C:\Users\JiunYong_Liang\AppData\Local\Temp\tm_report_copy.xlsx"
shutil.copy2(EXCEL_FILE, dst)

df = pd.read_excel(dst, sheet_name=SHEET_NAME, header=2)
mask = df["URL / Filename"].astype(str).str.startswith("http")
data_all = df[mask].reset_index(drop=True)

data = data_all.iloc[START_ROW:END_ROW].reset_index(drop=True)

print(f"Rows to process: {len(data)}")
data[["Course Code", "Learning Object Name", "URL / Filename"]]

Rows to process: 40


,Course Code,Learning Object Name,URL / Filename
0,MR-8WN-SLD01387,CSG: Sustainable Devices Sales Overview,https://edutube.dell.com/html5/videoPlayer.htm...
1,MR-8WN-SLD03887,Architecting Scalable AI: Inside the Dell AI D...,https://edutube.dell.com/html5/videoPlayer.htm...
2,MR-8WN-SLD04353,AI Data Platform - Understanding Use Cases,https://edutube.dell.com/html5/videoPlayer.htm...
3,MR-8WN-SLD04335,Dell AI Factory with Nvidia Technical Overview,https://edutube.dell.com/html5/videoPlayer.htm...
4,MR-8WN-SLD04343,Dell AI Data Platform - How to Sell and Position,https://edutube.dell.com/html5/videoPlayer.htm...
5,MR-8WN-SLD02239,PowerMax Sizing Basics,https://edutube.dell.com/html5/videoPlayer.htm...
6,MR-8WN-SLD04344,Storage and Private Cloud Positioning Guidance...,https://edutube.dell.com/html5/videoPlayer.htm...
7,MR-8WN-CSGT1003,Cloud Client Workspaces Technical Overview,https://edutube.dell.com/html5/videoPlayer.htm...
8,MR-8WN-SLD04356,Dell Private Cloud deploying RHOS Technical Re...,https://edutube.dell.com/html5/videoPlayer.htm...
9,MR-8WN-SLD04367,Dell Technologies Advantage for Partners: AI,https://edutube.dell.com/html5/videoPlayer.htm...


In [9]:
# CELL 4 - Open Chrome (run once, then login in the browser)
import subprocess
subprocess.Popen([
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    "--remote-debugging-port=9222",
    f"--user-data-dir={SESSION_DIR}",
    f"{DOCEBO_BASE}/learn/signin"
])
print("Chrome opened! Login, then run Cell 5.")

Chrome opened! Login, then run Cell 5.


In [10]:
# CELL 5 - Full automation loop
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

STEP_WAIT = 2

def wait_for_load():
    WebDriverWait(driver, 20).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(STEP_WAIT)

def do_search(course_code):
    """Navigate to course manage, search by code, verify results. Returns True if found."""
    driver.get(f"{DOCEBO_BASE}/course/manage")
    wait_for_load()
    search = wait.until(EC.presence_of_element_located((By.ID, "search-2")))
    # Clear with JS + trigger Angular input event
    driver.execute_script("""
        arguments[0].value = '';
        arguments[0].dispatchEvent(new Event('input', {bubbles: true}));
    """, search)
    time.sleep(1)
    search.send_keys(course_code)
    driver.execute_script("arguments[0].dispatchEvent(new Event('input', {bubbles: true}))", search)
    search.send_keys("\n")
    time.sleep(STEP_WAIT + 2)
    wait_for_load()

    # Retry up to 3 times waiting for correct result
    for attempt in range(3):
        try:
            WebDriverWait(driver, 10).until(
                lambda d: course_code in d.find_element(By.CSS_SELECTOR, "table tbody").text
            )
            return True
        except:
            if attempt < 2:
                print(f"  Search not ready yet, retrying ({attempt+1}/3)...")
                time.sleep(STEP_WAIT + 2)

    # Still not found — print what IS in the table for debugging
    try:
        table_text = driver.find_element(By.CSS_SELECTOR, "table tbody").text.strip()
        print(f"  Table shows: {table_text[:300] if table_text else '(empty)'}")
    except:
        print(f"  Table not found on page")
    return False

def process_row(course_code, title, url, idx, total):
    print(f"[{idx+1}/{total}] {course_code} | {title[:50]}")
    try:
        # Search and verify
        if not do_search(course_code):
            raise Exception(f"Course {course_code} not found in search results — skipping")

        course_link = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.no-renderer")))
        course_href = course_link.get_attribute("href")
        course_id = course_href.strip("/").split("/")[-1].split(";")[0]

        # Go to training materials tab
        driver.get(f"{DOCEBO_BASE}/course/edit/{course_id};tab=training_materials")
        wait_for_load()

        # Click HTML page inside iframe
        iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
        driver.switch_to.frame(iframe)
        time.sleep(STEP_WAIT)
        html_btn = driver.find_element(By.CSS_SELECTOR, "a[data-lo-type='htmlpage']")
        driver.execute_script("arguments[0].click()", html_btn)
        print("  Clicked HTML page")
        wait_for_load()

        # Fill title
        title_input = wait.until(EC.presence_of_element_located((By.ID, "LearningHtmlpage_title")))
        title_input.clear()
        title_input.send_keys(title)
        print("  Title filled")
        time.sleep(STEP_WAIT)

        # Type in TinyMCE
        tinymce = driver.find_element(By.ID, "player-arena-htmlpage-editor-textarea_ifr")
        driver.switch_to.frame(tinymce)
        body = driver.find_element(By.TAG_NAME, "body")
        body.click()
        body.send_keys("Click link: ")
        driver.switch_to.parent_frame()
        print("  TinyMCE typed")
        time.sleep(STEP_WAIT)

        # Click link button
        link_btn = driver.find_element(By.CSS_SELECTOR, "i.mce-i-link")
        link_btn.click()
        time.sleep(STEP_WAIT)

        # Fill URL
        url_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".mce-window .mce-textbox")))
        url_input.clear()
        url_input.send_keys(url)
        print("  URL filled")
        time.sleep(STEP_WAIT)

        # Set target to New window
        target_btn = driver.find_element(By.XPATH, "//button[contains(., 'None')]")
        target_btn.click()
        time.sleep(STEP_WAIT)
        new_window = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='menuitem'][contains(., 'New window')]")
        ))
        new_window.click()
        print("  Target set")
        time.sleep(STEP_WAIT)

        # Click OK
        ok_btn = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//div[contains(@class,'mce-window')]//button[normalize-space(.)='Ok']")
        ))
        driver.execute_script("arguments[0].click()", ok_btn)
        print("  OK clicked")
        time.sleep(STEP_WAIT)

        # Click Save changes
        save_btn = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "input[name='confirm_save_htmlpage']")
        ))
        driver.execute_script("arguments[0].click()", save_btn)
        print("  Saved!")
        wait_for_load()

        # Click ico-menu
        menu_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "i.ico-menu")))
        driver.execute_script("arguments[0].click()", menu_btn)
        print("  Menu opened")
        time.sleep(STEP_WAIT)

        # Click "Upload to the Training material library"
        upload_btn = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[contains(., 'Upload to the Training material library')]")
        ))
        driver.execute_script("arguments[0].click()", upload_btn)
        print("  Upload panel opened")
        wait_for_load()

        # Switch to main content (panel is outside iframe)
        driver.switch_to.default_content()
        time.sleep(STEP_WAIT)

        # Click "Dell Learning" folder
        dell_learning = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//span[contains(@class,'ui-typography-heading-5') and contains(text(),'Dell Learning')]")
        ))
        driver.execute_script("arguments[0].click()", dell_learning)
        print("  Dell Learning selected!")
        time.sleep(STEP_WAIT)

        # Click Upload button
        upload_final = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(@class,'ui-button-color-accent') and contains(., 'Upload')]")
        ))
        driver.execute_script("arguments[0].click()", upload_final)
        print("  Done!")
        wait_for_load()
        return True

    except Exception as e:
        print(f"  ERROR: {e}")
        driver.switch_to.default_content()
        return False

success, failed, failed_rows = 0, 0, []
total = len(data)

for idx, row in data.iterrows():
    course_code = str(row["Course Code"]).strip()
    title = str(row["Learning Object Name"]).strip()
    url = str(row["URL / Filename"]).strip()
    if not course_code or course_code == "nan":
        continue
    ok = process_row(course_code, title, url, idx, total)
    if ok:
        success += 1
    else:
        failed += 1
        failed_rows.append(course_code)

print(f"\nDone! Success: {success} | Failed: {failed}")
if failed_rows:
    print("Failed:", ", ".join(failed_rows))

[1/40] MR-8WN-SLD01387 | CSG: Sustainable Devices Sales Overview
  Clicked HTML page
  Title filled
  TinyMCE typed
  URL filled
  Target set
  OK clicked
  Saved!
  Menu opened
  Upload panel opened
  Dell Learning selected!
  Done!
[2/40] MR-8WN-SLD03887 | Architecting Scalable AI: Inside the Dell AI Data 
  Clicked HTML page
  Title filled
  TinyMCE typed
  URL filled
  Target set
  OK clicked
  Saved!
  Menu opened
  Upload panel opened
  Dell Learning selected!
  Done!
[3/40] MR-8WN-SLD04353 | AI Data Platform - Understanding Use Cases
  Clicked HTML page
  Title filled
  TinyMCE typed
  URL filled
  Target set
  OK clicked
  Saved!
  Menu opened
  Upload panel opened
  Dell Learning selected!
  Done!
[4/40] MR-8WN-SLD04335 | Dell AI Factory with Nvidia Technical Overview
  Clicked HTML page
  Title filled
  TinyMCE typed
  URL filled
  Target set
  OK clicked
  Saved!
  Menu opened
  Upload panel opened
  Dell Learning selected!
  Done!
[5/40] MR-8WN-SLD04343 | Dell AI Data Platf

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 6 - Debug: test clicking on the current open browser page
# 1. Run Cell 4 to open browser and login
# 2. Manually navigate to the Training material tab
# 3. Then run THIS cell to test clicking the button
# ============================================================
import asyncio, threading
from playwright.async_api import async_playwright

async def test_click():
    async with async_playwright() as p:
        ctx = await p.chromium.launch_persistent_context(
            SESSION_DIR,
            headless=False,
            slow_mo=500,
            viewport={"width": 1400, "height": 900}
        )
        page = ctx.pages[0] if ctx.pages else await ctx.new_page()
        print(f"Current URL: {page.url}")

        # List all links on current page
        links = await page.evaluate("""
            () => Array.from(document.querySelectorAll('a'))
                .map(a => ({text: a.textContent.trim(), class: a.className, href: a.getAttribute('href')}))
                .filter(a => a.text.length > 0)
        """)
        print("Links found:")
        for l in links:
            if 'Add' in l['text'] or 'training' in l['text'].lower() or 'material' in l['text'].lower():
                print(f"  >>> {l}")

        # Try clicking Add training material
        print("\nTrying to click Add training material...")
        btn = page.locator('a.btn-docebo')
        count = await btn.count()
        print(f"  Found {count} a.btn-docebo elements")

        if count > 0:
            for i in range(count):
                txt = await btn.nth(i).text_content()
                print(f"  [{i}] text: {txt.strip()}")

            await btn.first.click(force=True)
            print("  Clicked! Check the browser.")
        else:
            print("  No btn-docebo found. Try scrolling down in the browser first.")

        await asyncio.sleep(5)
        await ctx.close()

def run_test():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(test_click())
    loop.close()

threading.Thread(target=run_test).start()

In [ ]:
# ============================================================
# CELL 6 - Open browser and go to Training material page
# Run this, then manually navigate if needed, then run Cell 7
# ============================================================
import asyncio, threading
from playwright.async_api import async_playwright

_state = {}

async def open_browser():
    pw = await async_playwright().start()
    ctx = await pw.chromium.launch_persistent_context(
        SESSION_DIR,
        headless=False,
        slow_mo=300,
        viewport={"width": 1400, "height": 900}
    )
    page = ctx.pages[0] if ctx.pages else await ctx.new_page()
    await page.goto("https://dellsandbox.docebosaas.com/course/edit/2564;tab=training_materials")
    await page.wait_for_load_state("networkidle")
    await page.wait_for_timeout(3000)

    _state['pw'] = pw
    _state['ctx'] = ctx
    _state['page'] = page
    print("Browser opened! Navigate to the Training material tab if needed.")
    print("Then run Cell 7 to test clicking the button.")

def run():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(open_browser())
    loop.close()

threading.Thread(target=run).start()

In [ ]:
# ============================================================
# CELL 7 - Test: click "Add training material" on open browser
# Make sure you're on the Training material tab first!
# ============================================================
import asyncio, threading
from playwright.async_api import async_playwright

async def test_add_btn():
    async with async_playwright() as p:
        ctx = await p.chromium.launch_persistent_context(
            SESSION_DIR,
            headless=False,
            slow_mo=500,
            viewport={"width": 1400, "height": 900}
        )
        page = ctx.pages[0] if ctx.pages else await ctx.new_page()
        print(f"URL: {page.url}")

        # Find all matching buttons and print info
        info = await page.evaluate("""
            () => Array.from(document.querySelectorAll('a'))
                .filter(a => a.textContent.trim().includes('Add training material'))
                .map(a => ({
                    text: a.textContent.trim(),
                    className: a.className,
                    visible: a.offsetParent !== null,
                    rect: JSON.stringify(a.getBoundingClientRect())
                }))
        """)
        print(f"Found {len(info)} 'Add training material' buttons:")
        for i, btn in enumerate(info):
            print(f"  [{i}] {btn}")

        if not info:
            print("Button NOT found in DOM. Page may not have loaded the tab content yet.")
            await ctx.close()
            return

        # Try clicking with real mouse coordinates
        btn_locator = page.locator('a.btn-docebo')
        box = await btn_locator.first.bounding_box()
        if box:
            print(f"\nBounding box: {box}")
            print("Clicking with real mouse coordinates...")
            await page.mouse.click(box['x'] + box['width']/2, box['y'] + box['height']/2)
            await page.wait_for_timeout(2000)
            print("Done! Check browser if dropdown opened.")
        else:
            print("No bounding box - element not visible on screen")

        await asyncio.sleep(5)
        await ctx.close()

def run_test():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(test_add_btn())
    loop.close()

threading.Thread(target=run_test).start()

In [ ]:
# CELL 8 - 开 Chrome (先关掉现有的 Chrome，然后跑这个)
import subprocess
subprocess.Popen([
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    "--remote-debugging-port=9222",
    f"--user-data-dir={SESSION_DIR}",
    "https://dellsandbox.docebosaas.com/course/edit/2564;tab=training_materials"
])
print("Chrome opened! Login, go to the URL, then run Cell 9.")

In [ ]:
# CELL 9 - Test full flow: HTML page form
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

driver.get("https://dellsandbox.docebosaas.com/course/edit/2564;tab=training_materials")
time.sleep(3)

# Switch into iframe, click HTML page
iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
driver.switch_to.frame(iframe)
time.sleep(1)
html_btn = driver.find_element(By.CSS_SELECTOR, "a[data-lo-type='htmlpage']")
driver.execute_script("arguments[0].click()", html_btn)
print("Step 1: Clicked HTML page")
time.sleep(3)

# Switch back to main page (form is outside iframe)
driver.switch_to.default_content()
time.sleep(1)

# Step 2: Fill title
TEST_TITLE = "TEST - Enterprise SONiC Distribution"
TEST_URL = "https://example.com/test"

title_input = wait.until(EC.presence_of_element_located((By.ID, "LearningHtmlpage_title")))
title_input.clear()
title_input.send_keys(TEST_TITLE)
print("Step 2: Title filled")

# Step 3: Type in TinyMCE
tinymce = driver.find_element(By.ID, "player-arena-htmlpage-editor-textarea_ifr")
driver.switch_to.frame(tinymce)
body = driver.find_element(By.TAG_NAME, "body")
body.click()
body.send_keys("Click link: ")
driver.switch_to.default_content()
print("Step 3: Typed in TinyMCE")

# Step 4: Click link button in toolbar
link_btn = driver.find_element(By.CSS_SELECTOR, "i.mce-i-link")
link_btn.click()
time.sleep(1)
print("Step 4: Clicked link button")

# Step 5: Fill URL
url_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".mce-window .mce-textbox")))
url_input.clear()
url_input.send_keys(TEST_URL)
print("Step 5: URL filled")

# Step 6: Set target to New window
target_btn = driver.find_element(By.XPATH, "//button[contains(., 'None')]")
target_btn.click()
time.sleep(0.5)
new_window = wait.until(EC.element_to_be_clickable((By.XPATH, "//li[contains(., 'New window')]")))
new_window.click()
print("Step 6: Target set to New window")

# Step 7: Click OK
ok_btn = driver.find_element(By.XPATH, "//button[contains(., 'Ok')]")
ok_btn.click()
print("Step 7: OK clicked!")
time.sleep(2)
print("All steps done! Check Chrome.")